# Data Retrieval Pipeline — Finland Weather Predictor (V2)

This notebook builds the historical training dataset for the V2 weather prediction models. It fetches raw observations from the **[Finnish Meteorological Institute (FMI)](https://en.ilmatieteenlaitos.fi/)** via two WFS endpoints across **10 stations in 6 Finnish cities**, spanning **~2.3 years** of hourly data (Nov 2023 → Mar 2026).

---

## Data Sources

| Endpoint | Parameters | Resolution | API Limit |
|----------|------------|------------|-----------|
| `fmi::observations::weather::hourly` | temp, humidity, wind speed/dir/gust, pressure, precipitation, WAWA code | 1h | 744h per request |
| `fmi::observations::weather::multipointcoverage` | cloud cover, dew point, snow depth | 10-min → resampled to 1h | 168h per request |

## Stations

| City | Station(s) | Type |
|------|------------|------|
| Helsinki | Kumpula | Coastal |
| Oulu | Airport, Kaukovainio | Coastal |
| Tampere | Airport, Harmala | Inland |
| Turku | Artukainen | Coastal |
| Rovaniemi | Apukka, Airport | Arctic inland |
| Vaasa | Klemettila, Airport | Coastal |

## Fetch Strategy

- **28 monthly chunks** to cover the full date range within FMI API request limits
- **Parallel fetching** via `ThreadPoolExecutor` — both endpoints run concurrently per station per chunk
- **3× retry with 5s backoff** on 10-min sub-chunks to handle transient API failures
- 10-min timestamps are **floored to the nearest hour** then averaged before merging with hourly data on `fmisid + time`

## Output

Saves `updated_weather_2y.csv` — ~165,000 raw station-hour rows, used directly as input to:
- [`temp_forecast.ipynb`](temp_forecast.ipynb) — XGBoost temperature regression
- [`code_forecast.ipynb`](code_forecast.ipynb) — LightGBM weather code classifier

In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from fmiopendata.wfs import download_stored_query
from concurrent.futures import ThreadPoolExecutor, as_completed

reliable_stations = {
    "101786": {"city": "Oulu",      "name": "Oulu Airport",       "lat": 64.93, "lon": 25.35},
    "108040": {"city": "Oulu",      "name": "Oulu Kaukovainio",   "lat": 65.01, "lon": 25.47},
    "101004": {"city": "Helsinki",  "name": "Helsinki Kumpula",   "lat": 60.20, "lon": 24.96},
    "101118": {"city": "Tampere",   "name": "Tampere Airport",    "lat": 61.41, "lon": 23.60},
    "101124": {"city": "Tampere",   "name": "Tampere Harmala",    "lat": 61.47, "lon": 23.75},
    "100949": {"city": "Turku",     "name": "Turku Artukainen",   "lat": 60.51, "lon": 22.20},
    "101933": {"city": "Rovaniemi", "name": "Rovaniemi Apukka",   "lat": 66.56, "lon": 25.83},
    "137190": {"city": "Rovaniemi", "name": "Rovaniemi Airport",  "lat": 66.56, "lon": 25.83},
    "101485": {"city": "Vaasa",     "name": "Vaasa Klemettila",   "lat": 63.09, "lon": 21.65},
    "101462": {"city": "Vaasa",     "name": "Vaasa Airport",      "lat": 63.05, "lon": 21.76},
}

# Hourly endpoint parameters
PARAMETERS_HOURLY = "TA_PT1H_AVG,TA_PT1H_MAX,TA_PT1H_MIN,RH_PT1H_AVG,WS_PT1H_AVG,WS_PT1H_MAX,WD_PT1H_AVG,WG_PT1H_MAX,PA_PT1H_AVG,PRA_PT1H_ACC,WAWA_PT1H_RANK"

# 10-min endpoint parameters
PARAMETERS_10MIN  = "vis,n_man,td,snow_aws"

PARAM_MAP_HOURLY = {
    "air temperature":          "temp",
    "maximum temperature":      "highest_temperature",
    "minimum temperature":      "lowest_temperature",
    "relative humidity":        "humidity",
    "wind speed":               "wind_speed",
    "maximum wind speed":       "wind_speed_max",
    "wind direction":           "wind_dir",
    "wind gust":                "greatest_gust_speed",
    "air pressure":             "pressure_sea",
    "precipitation amount":     "precip_1h",
    "present weather (auto)":   "present_weather_(auto)",
}

PARAM_MAP_10MIN = {
    "cloud amount":           "cloud_cover",
    "dew-point temperature":  "dew_point",
    "snow depth":             "snow_depth",
}

MAX_WORKERS = 4

In [2]:
def fetch_station_hourly(fmisid, info, start, end):
    """Fetch hourly FMI observations for one station."""
    starttime_str = start.strftime('%Y-%m-%dT%H:%M:%SZ')
    endtime_str   = end.strftime('%Y-%m-%dT%H:%M:%SZ')

    try:
        obs = download_stored_query(
            "fmi::observations::weather::hourly::multipointcoverage",
            args=[
                f"fmisid={fmisid}",
                f"starttime={starttime_str}",
                f"endtime={endtime_str}",
                f"parameters={PARAMETERS_HOURLY}",
            ]
        )

        if not obs.data:
            return fmisid, None, "no data"

        rows = []
        for timestamp, stations in obs.data.items():
            for station_name, params in stations.items():
                row = {
                    "time":         timestamp,
                    "city":         info["city"],
                    "station_name": info["name"],
                    "fmisid":       fmisid,
                    "lat":          info["lat"],
                    "lon":          info["lon"],
                }
                for p_name, p_data in params.items():
                    col = PARAM_MAP_HOURLY.get(p_name.lower(), p_name.replace(" ", "_").lower())
                    row[col] = p_data.get("value")
                rows.append(row)

        if rows:
            return fmisid, pd.DataFrame(rows), f"{len(rows)} rows"
        return fmisid, None, "empty"

    except Exception as e:
        return fmisid, None, f"✖ {e}"

In [3]:
import time

def fetch_station_10min(fmisid, info, start, end):
    """
    Fetch 10-min FMI observations in 7-day sub-chunks.
    """
    sub_chunks = []
    sub_start = start
    while sub_start < end:
        sub_end = min(sub_start + timedelta(days=7), end)
        sub_chunks.append((sub_start, sub_end))
        sub_start = sub_end

    all_rows = []

    for sub_start, sub_end in sub_chunks:
        for attempt in range(3):
            try:
                obs = download_stored_query(
                    "fmi::observations::weather::multipointcoverage",
                    args=[
                        f"fmisid={fmisid}",
                        f"starttime={sub_start.strftime('%Y-%m-%dT%H:%M:%SZ')}",
                        f"endtime={sub_end.strftime('%Y-%m-%dT%H:%M:%SZ')}",
                        f"parameters={PARAMETERS_10MIN}",
                    ]
                )

                if not obs.data:
                    break  # no data but not an error — move to next sub-chunk

                for timestamp, stations in obs.data.items():
                    for station_name, params in stations.items():
                        row = {
                            "time":   timestamp,
                            "fmisid": fmisid,
                            "city":   info["city"],
                        }
                        for p_name, p_data in params.items():
                            col = PARAM_MAP_10MIN.get(p_name.lower(), p_name.replace(" ", "_").lower())
                            row[col] = p_data.get("value")
                        all_rows.append(row)

                break 

            except Exception as e:
                if attempt < 2:
                    time.sleep(5)  # wait before retrying
                else:
                    print(f"    ✖ 10min sub-chunk {sub_start.date()}→{sub_end.date()} ({fmisid}): {e}")

    if not all_rows:
        return fmisid, None, "empty"

    df = pd.DataFrame(all_rows)
    df["time"] = pd.to_datetime(df["time"])
    df = df.sort_values("time")

    # Floor to nearest hour
    df["time"] = df["time"].dt.floor("h")

    # Only aggregate columns that actually exist in this station's data
    POSSIBLE_COLS = {
        "cloud_cover": "mean",
        "dew_point":   "mean",
        "snow_depth":  "last",
    }

    agg_cols = {col: agg for col, agg in POSSIBLE_COLS.items() if col in df.columns}

    if not agg_cols:
        return fmisid, None, "no usable columns"

    agg_cols["city"] = "first"

    df_hourly = (
        df.groupby(["fmisid", "time"])
        .agg(agg_cols)
        .reset_index()
    )

    return fmisid, df_hourly, f"{len(df_hourly)} hourly rows"

In [4]:
now    = datetime(2026, 3, 5, 0, 0, 0) 

# Getting 28 months, so about more than 2 years worth of data 
# Oulu Kaukovainio begin to operate around the end of 2023, so this is our limit
chunks = [
    (now - timedelta(days=(i + 1) * 30), now - timedelta(days=i * 30))
    for i in range(28)
]

print(f"Total chunks: {len(chunks)} | {chunks[-1][0].date()} → {chunks[0][1].date()}")

all_chunks_dfs = []

for count, (start, end) in enumerate(chunks, start=1):
    hourly_dfs = []
    min10_dfs  = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {}
        for fmisid, info in reliable_stations.items():
            futures[executor.submit(fetch_station_hourly, fmisid, info, start, end)] = ("hourly", fmisid, info)
            futures[executor.submit(fetch_station_10min,  fmisid, info, start, end)] = ("10min",  fmisid, info)

        warnings = []
        for future in as_completed(futures):
            source, fmisid, info = futures[future]
            _, df, status        = future.result()

            if df is not None:
                if source == "hourly": hourly_dfs.append(df)
                else:                  min10_dfs.append(df)
            else:
                warnings.append(f"{info['name']} [{source}]: {status}")

    if not hourly_dfs:
        print(f"  CHUNK {count:>2} ✖ no hourly data — skipped")
        continue

    df_fmi         = pd.concat(hourly_dfs, ignore_index=True)
    df_fmi["time"] = pd.to_datetime(df_fmi["time"]).dt.floor("h")

    if min10_dfs:
        df_10min = pd.concat(min10_dfs, ignore_index=True)
        for col in ["cloud_cover", "dew_point", "snow_depth"]:
            if col not in df_10min.columns:
                df_10min[col] = np.nan

        df_chunk = df_fmi.merge(
            df_10min[["fmisid", "time", "cloud_cover", "dew_point", "snow_depth"]],
            on=["fmisid", "time"],
            how="left"
        )
    else:
        df_chunk = df_fmi

    all_chunks_dfs.append(df_chunk)


    warn_str = f" | ! {len(warnings)} warnings" if warnings else ""
    print(f"  CHUNK {count:>2}/{len(chunks)} ✅ {len(df_chunk):,} rows{warn_str}")
    for w in warnings:
        print(f"    ↳ {w}")

Total chunks: 28 | 2023-11-16 → 2026-03-05
  CHUNK  1/28 ✅ 7,205 rows
  CHUNK  2/28 ✅ 7,145 rows
  CHUNK  3/28 ✅ 7,210 rows
  CHUNK  4/28 ✅ 7,207 rows
  CHUNK  5/28 ✅ 7,206 rows
  CHUNK  6/28 ✅ 7,208 rows
  CHUNK  7/28 ✅ 7,210 rows
  CHUNK  8/28 ✅ 7,210 rows
  CHUNK  9/28 ✅ 7,209 rows
  CHUNK 10/28 ✅ 7,197 rows
  CHUNK 11/28 ✅ 7,169 rows
  CHUNK 12/28 ✅ 7,207 rows
  CHUNK 13/28 ✅ 7,208 rows
  CHUNK 14/28 ✅ 7,208 rows
  CHUNK 15/28 ✅ 7,196 rows
  CHUNK 16/28 ✅ 7,203 rows
  CHUNK 17/28 ✅ 7,209 rows
  CHUNK 18/28 ✅ 7,210 rows
  CHUNK 19/28 ✅ 7,209 rows
  CHUNK 20/28 ✅ 7,210 rows
  CHUNK 21/28 ✅ 7,210 rows
  CHUNK 22/28 ✅ 7,210 rows
  CHUNK 23/28 ✅ 7,208 rows
  CHUNK 24/28 ✅ 7,203 rows
  CHUNK 25/28 ✅ 7,210 rows
  CHUNK 26/28 ✅ 7,210 rows
  CHUNK 27/28 ✅ 7,179 rows
  CHUNK 28/28 ✅ 7,210 rows


In [5]:
df_raw = pd.concat(all_chunks_dfs, ignore_index=True)
df_raw = df_raw.sort_values(["time", "fmisid"]).reset_index(drop=True)
df_raw.to_csv("updated_weather_2y.csv", index=False)

print(f"✅ Saved — {len(df_raw):,} rows | {df_raw.shape[1]} columns")
print(f"\nColumns:\n{list(df_raw.columns)}")
print(f"\nNull summary:")
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])

✅ Saved — 201,676 rows | 20 columns

Columns:
['time', 'city', 'station_name', 'fmisid', 'lat', 'lon', 'temp', 'highest_temperature', 'lowest_temperature', 'humidity', 'wind_speed', 'wind_speed_max', 'wind_dir', 'greatest_gust_speed', 'pressure_sea', 'precip_1h', 'present_weather_(auto)', 'cloud_cover', 'dew_point', 'snow_depth']

Null summary:
temp                         71
highest_temperature          48
lowest_temperature           45
humidity                   1130
wind_speed                41641
wind_speed_max            42434
wind_dir                  42395
greatest_gust_speed       41753
pressure_sea              20147
precip_1h                 60589
present_weather_(auto)    40351
cloud_cover               20285
dew_point                    11
snow_depth                60896
dtype: int64


In [6]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 201676 entries, 0 to 201675
Data columns (total 20 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   time                    201676 non-null  datetime64[us]
 1   city                    201676 non-null  str           
 2   station_name            201676 non-null  str           
 3   fmisid                  201676 non-null  str           
 4   lat                     201676 non-null  float64       
 5   lon                     201676 non-null  float64       
 6   temp                    201605 non-null  float64       
 7   highest_temperature     201628 non-null  float64       
 8   lowest_temperature      201631 non-null  float64       
 9   humidity                200546 non-null  float64       
 10  wind_speed              160035 non-null  float64       
 11  wind_speed_max          159242 non-null  float64       
 12  wind_dir                159281 non-null  